In [1]:
%run n50_prep.ipynb

print(n50.head())

                        Open
datetime                    
2015-01-09 09:15:00  8285.45
2015-01-09 09:16:00  8292.60
2015-01-09 09:17:00  8287.40
2015-01-09 09:18:00  8294.25
2015-01-09 09:19:00  8300.60

Shape:  (852087, 1)

Shape after filtering market hours:  (851460, 1)
time          09:15    09:16    09:17    09:18   09:19    09:20    09:21  \
date                                                                       
2015-01-09  8285.45  8292.60  8287.40  8294.25  8300.6  8300.50  8300.65   
2015-01-12  8291.35  8254.20  8255.25  8258.15  8263.2  8267.45  8266.05   
2015-01-13  8346.15  8355.15  8348.70  8344.50  8342.5  8340.35  8339.75   
2015-01-14  8307.25  8300.85  8307.00  8309.05  8305.4  8304.70  8302.20   
2015-01-15  8425.20  8440.45  8394.35  8386.05  8401.1  8428.00  8408.25   

time          09:22    09:23    09:24  ...   15:20    15:21    15:22    15:23  \
date                                   ...                                      
2015-01-09  8302.45  8294.85  8

In [2]:
# Backup original data
n50_orig = n50.copy()
n50_orig["mean"] = n50.mean(axis=1)
n50_orig["std"] = n50.std(axis=1)

 - Standardize the data
 - Scale the target value also as part of standardization
 - Take out every 9th day as test set

In [3]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Convert n50 into numpy array for processing
X = n50.values

# Now X is a 2D array.
# Every row is corresponding to a sample (a date)
# Every column is a corresponding to the price at a particular time.
# We shouldn't standardize the column wise data here. It is not a feature. We should standardize only row wise data
# But StandardScalar expects the features to be in columns.
# So, first transpose X and then standardize, then transpose back
X = X.T
X = scaler.fit_transform(X)
X = X.T

print(X[:5])

import pandas as pd

n50_std = pd.DataFrame(X, columns=n50.columns, index=n50.index)

[[ 1.04469641  1.3085318   1.11665152 ... -0.66746611 -0.63241106
   0.56868571]
 [ 1.09287207 -2.04993408 -1.96110645 ...  3.27549384  3.45737899
   3.30087316]
 [ 0.99096459  2.16998083  1.32501919 ... -1.66837203 -2.755687
  -5.28402182]
 [ 1.13100347  0.84874476  1.11997774 ...  0.17397003  0.16073916
  -0.40377826]
 [-0.77460424 -0.25926211 -1.81711602 ...  1.38645342  1.46924609
   2.04879479]]


Dataset is almost ready. Let's remove some outlier days. Somedays would have had some big news that affected the market significantly. Let's remove any row from the dataset where the swing is huge.

How to do it?

 - Calculate sd for each date
 - Calculate sd of all SDs
 - Remove any row with more than 2 SDs

**Note**: *This is a questionable approach. If a day is not very volatile but by end of the day suddenly the index dropped, that may not be removed from this method. Other approach can be scan each row for any outlier. Something like any value which is swung more than 10 std or more. We can find it only trial and error.*

In [4]:
n50_std["std"] = n50_std.std(axis=1)

mean_of_std = n50_std["std"].mean()
std_of_std = n50_std["std"].std()

print("Mean of std:", mean_of_std)
print("Std of std:", std_of_std)

n50_std.drop("std", axis=1, inplace=True)


Mean of std: 1.0015810286552085
Std of std: 5.769173951960499e-16


Standard deviation of standard deviations is very low. And for some reason, it is not giving good confidence. Let's manually remove any row with higher swing.

In [5]:
# Now remove any row with more than 5% swing
n50_cleaned = n50_std[(n50_std.max(axis=1) - n50_std.min(axis=1)) <= 9]

print("Original data size:", n50.shape)
print("Cleaned data size:", n50_cleaned.shape)

# Print Removed dates
removed_dates = n50_std.index.difference(n50_cleaned.index)
print("Removed dates due to high swing:")
for date in removed_dates:
    print(date)

Original data size: (2243, 317)
Cleaned data size: (2226, 317)
Removed dates due to high swing:
2015-03-04
2015-06-10
2016-01-15
2016-02-08
2016-06-21
2016-07-20
2016-10-26
2017-08-23
2017-12-27
2018-01-01
2019-08-08
2020-04-03
2020-05-13
2020-07-17
2020-10-06
2021-01-21
2023-04-17


In [6]:
# X = n50_cleaned.drop(columns=["target"]).values
# y = n50_cleaned["target"].values

X = n50_cleaned.drop(columns=["target"])
y = n50_cleaned["target"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)
print("RMSE:", rmse)

print("Sample Predictions:" )

for i in range(5):
    print(f"Predicted: {y_pred[i]}, Actual: {y_test[i]}")
    predicted_value = y_pred[i] * n50_orig.loc[X_test.index[i], 'std'] + n50_orig.loc[X_test.index[i], 'mean']
    print(f"Original Value: {n50_orig.loc[X_test.index[i], 'target']}, Predicted Value: {predicted_value} \n")




Train shape: (1780, 316)
Test shape: (446, 316)
RMSE: 1.3547788341873972e-10
Sample Predictions:
Predicted: 1.1253212702448117, Actual: 1.125321269973269
Original Value: 8769.35, Predicted Value: 8769.365966646243 

Predicted: -2.2990128950972006, Actual: -2.299012895392507
Original Value: 8752.05, Predicted Value: 8752.02490079808 

Predicted: -3.14065499140778, Actual: -3.140654991455343
Original Value: 18314.9, Predicted Value: 18314.811437209424 

Predicted: 3.4954314169936134, Actual: 3.49543141696352
Original Value: 7834.0, Predicted Value: 7834.11140167364 

Predicted: -0.3573480736288379, Actual: -0.35734807357389936
Original Value: 10361.9, Predicted Value: 10361.878789525967 



/tmp/ipykernel_2342606/2756783504.py:29: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Predicted: {y_pred[i]}, Actual: {y_test[i]}")
